# ResNet50 Transfer Learning for Leaf Disease Classification

This notebook implements ResNet50 (Residual Network 50) for 40-class leaf disease classification. 
ResNet50 is a deep CNN architecture that uses residual connections to enable training of very deep networks.
Perfect for leveraging ImageNet pretrained weights for improved feature extraction on agricultural images.

## 1. Import Required Libraries

In [1]:
import os
import sys
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Scikit-learn
from sklearn.metrics import (confusion_matrix, classification_report, 
                            accuracy_score, precision_score, recall_score, 
                            f1_score, roc_auc_score, roc_curve, auc)

# Add utils to path
sys.path.insert(0, '..')
from utils.model_utils import (
    build_resnet50_model,
    unfreeze_model_layers,
    train_model,
    evaluate_model,
    save_model,
    load_model,
    compute_class_weights_from_labels,
    create_data_augmentation
)
from utils.data_utils import load_preprocessed_data

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")


ImportError: cannot import name 'load_preprocessed_data' from 'utils.data_utils' (c:\Users\GAME-PC\Downloads\cnn_model\notebooks\..\utils\data_utils.py)

## 2. Configuration and Data Loading

In [ ]:
# Configuration
CONFIG = {
    'INPUT_SHAPE': (224, 224, 3),
    'NUM_CLASSES': 40,
    'BATCH_SIZE': 16,
    
    # Phase 1: Frozen base model (feature extraction)
    'EPOCHS_PHASE1': 20,
    'LEARNING_RATE_PHASE1': 0.0001,
    
    # Phase 2: Fine-tuning (selective layer unfreezing)
    'EPOCHS_PHASE2': 15,
    'LEARNING_RATE_PHASE2': 0.00001,
    'TRAINABLE_LAYERS': 50,
    
    # Architecture
    'DENSE_UNITS': 256,
    'DROPOUT_RATE': 0.35,
    'OPTIMIZER': 'adam',
    
    # Directories
    'DATA_DIR': '../results/preprocessed_data',
    'MODELS_DIR': '../models/resnet50',
    'RESULTS_DIR': '../results/resnet50_results'
}

# Create directories
os.makedirs(CONFIG['MODELS_DIR'], exist_ok=True)
os.makedirs(CONFIG['RESULTS_DIR'], exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")


In [ ]:
# Load preprocessed data
X_train = np.load(os.path.join(CONFIG['DATA_DIR'], 'X_train.npy'))
X_val = np.load(os.path.join(CONFIG['DATA_DIR'], 'X_val.npy'))
X_test = np.load(os.path.join(CONFIG['DATA_DIR'], 'X_test.npy'))

y_train = np.load(os.path.join(CONFIG['DATA_DIR'], 'y_train.npy'))
y_val = np.load(os.path.join(CONFIG['DATA_DIR'], 'y_val.npy'))
y_test = np.load(os.path.join(CONFIG['DATA_DIR'], 'y_test.npy'))

# Load label encoding
with open(os.path.join(CONFIG['DATA_DIR'], 'label_encoding.json'), 'r') as f:
    label_encoding = json.load(f)

class_names = sorted(label_encoding.keys(), key=lambda x: label_encoding[x])

print(f"\nData shapes:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"\nNumber of classes: {CONFIG['NUM_CLASSES']}")
print(f"Data range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"\nClass names ({len(class_names)} total):")
for i, name in enumerate(class_names[:10]):
    print(f"  {i:2d}: {name}")
if len(class_names) > 10:
    print(f"  ... and {len(class_names) - 10} more")


In [ ]:
# Compute class weights for imbalanced dataset
class_weight = compute_class_weights_from_labels(y_train, class_names=class_names, verbose=True)


## 3. Build ResNet50 Models with Custom Classification Heads

In [ ]:
# Build initial ResNet50 model (frozen base, high learning rate for feature extraction)
print("=" * 80)
print("PHASE 1: Building ResNet50 Model (Frozen Base, High Learning Rate)")
print("=" * 80)

resnet50_model = build_resnet50_model(
    num_classes=CONFIG['num_classes'],
    input_shape=CONFIG['input_shape'],
    learning_rate=CONFIG['phase1_learning_rate'],
    optimizer=CONFIG['optimizer'],
    dense_units=CONFIG['dense_units'],
    dropout_rate=CONFIG['dropout_rate']
)

print("\nResNet50 Model Summary:")
resnet50_model.summary()
print(f"\nTotal Parameters: {resnet50_model.count_params():,}")
print(f"Trainable Parameters: {sum([tf.size(w).numpy() for w in resnet50_model.trainable_weights]):,}")


## 4. Model Compilation Verification

In [ ]:
# Verify model compilation
print("\nModel Compilation Details:")
print(f"Optimizer: {resnet50_model.optimizer.__class__.__name__}")
print(f"Loss Function: {resnet50_model.loss}")
print(f"Metrics: {resnet50_model.metrics}")
print(f"\nPhase 1 Learning Rate: {CONFIG['phase1_learning_rate']}")
print(f"Phase 2 Learning Rate: {CONFIG['phase2_learning_rate']}")


## 5. Data Preprocessing and Augmentation

In [ ]:
# Verify input data shapes and ranges
print("Input Data Verification:")
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"X_train range: [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"y_test shape: {y_test.shape}")

# Create data augmentation pipeline for training
train_augmentation = keras.Sequential([
    layers.RandomRotation(0.2, input_shape=CONFIG['input_shape']),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomFlip("horizontal"),
    layers.GaussianNoise(0.02)
], name='augmentation')

print(f"\nData Augmentation Pipeline: {len(train_augmentation.layers)} layers")

# Prepare training data as tf.data.Dataset for efficient batching
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .shuffle(buffer_size=CONFIG['batch_size'] * 2) \
    .batch(CONFIG['batch_size']) \
    .prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)) \
    .batch(CONFIG['batch_size']) \
    .prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)) \
    .batch(CONFIG['batch_size']) \
    .prefetch(tf.data.AUTOTUNE)

print(f"\nDataset Pipeline Created:")
print(f"  Train batches: {len(train_dataset)}")
print(f"  Val batches: {len(val_dataset)}")
print(f"  Test batches: {len(test_dataset)}")


## 6. Define Training Callbacks

In [ ]:
# Create training callbacks for monitoring and early stopping
def create_callbacks(phase_name: str):
    """
    Create training callbacks for monitoring and early stopping.
    
    Args:
        phase_name: "phase1" or "phase2" to distinguish log files
    """
    callbacks = [
        # Early stopping to prevent overfitting
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        # Save best model based on validation accuracy
        keras.callbacks.ModelCheckpoint(
            filepath=f'{CONFIG["models_dir"]}/resnet50_{phase_name}_best.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=0
        ),
        # Reduce learning rate on plateau
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        # Log to tensorboard
        keras.callbacks.TensorBoard(
            log_dir=f'{CONFIG["logs_dir"]}/resnet50_{phase_name}',
            histogram_freq=1,
            write_graph=True
        )
    ]
    return callbacks

phase1_callbacks = create_callbacks('phase1')
print(f"Phase 1 Callbacks: {[cb.__class__.__name__ for cb in phase1_callbacks]}")


## 7. Phase 1 Training (Frozen Base, Feature Extraction)

In [ ]:
# Train ResNet50 model - Phase 1 (frozen base)
print("=" * 80)
print("PHASE 1 TRAINING: ResNet50 with Frozen Base")
print("=" * 80)
print(f"Epochs: {CONFIG['phase1_epochs']}")
print(f"Learning Rate: {CONFIG['phase1_learning_rate']}")
print(f"Batch Size: {CONFIG['batch_size']}")
print(f"Training Samples: {X_train.shape[0]}")
print("=" * 80)

resnet50_history = resnet50_model.fit(
    X_train, y_train,
    epochs=CONFIG['phase1_epochs'],
    batch_size=CONFIG['batch_size'],
    validation_data=(X_val, y_val),
    class_weight=class_weight,
    callbacks=phase1_callbacks,
    verbose=1
)

# Save training history
import json
history_dict = {
    'loss': [float(x) for x in resnet50_history.history['loss']],
    'accuracy': [float(x) for x in resnet50_history.history['accuracy']],
    'val_loss': [float(x) for x in resnet50_history.history['val_loss']],
    'val_accuracy': [float(x) for x in resnet50_history.history['val_accuracy']]
}

with open(f'{CONFIG["logs_dir"]}/resnet50_phase1_history.json', 'w') as f:
    json.dump(history_dict, f, indent=4)

print("\n" + "=" * 80)
print("PHASE 1 TRAINING COMPLETED")
print("=" * 80)
print(f"Final Train Loss: {resnet50_history.history['loss'][-1]:.4f}")
print(f"Final Train Accuracy: {resnet50_history.history['accuracy'][-1]:.4f}")
print(f"Final Val Loss: {resnet50_history.history['val_loss'][-1]:.4f}")
print(f"Final Val Accuracy: {resnet50_history.history['val_accuracy'][-1]:.4f}")


## 8. Evaluate Phase 1 Model Performance

In [ ]:
# Evaluate model on test set using model_utils
print("\n" + "=" * 80)
print("PHASE 1 EVALUATION ON TEST SET")
print("=" * 80)

metrics_phase1 = evaluate_model(
    resnet50_model,
    X_test,
    y_test,
    class_names=class_names
)

print("\nPhase 1 Test Set Metrics:")
print(f"  Accuracy: {metrics_phase1['accuracy']:.4f}")
print(f"  Macro Precision: {metrics_phase1['macro_precision']:.4f}")
print(f"  Macro Recall: {metrics_phase1['macro_recall']:.4f}")
print(f"  Macro F1-Score: {metrics_phase1['macro_f1']:.4f}")
print(f"  Weighted F1-Score: {metrics_phase1['weighted_f1']:.4f}")

# Get confusion matrix for later visualization
cm_phase1 = metrics_phase1['confusion_matrix']
print(f"\nConfusion Matrix shape: {cm_phase1.shape}")


## 9. Generate Confusion Matrix and Class-wise Performance

In [ ]:
# Save confusion matrix
import numpy as np
import json

# Save confusion matrix as numpy file
cm_phase1_path = f'{CONFIG["confusion_matrix_dir"]}/resnet50_phase1_cm.npy'
np.save(cm_phase1_path, cm_phase1)
print(f"Confusion matrix saved: {cm_phase1_path}")

# Calculate class-wise metrics from confusion matrix
def calculate_class_metrics(cm, class_names):
    """Calculate precision, recall, F1 for each class."""
    metrics = {}
    for i in range(len(class_names)):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        metrics[class_names[i]] = {
            'precision': float(precision),
            'recall': float(recall),
            'f1_score': float(f1),
            'support': int(cm[i, :].sum())
        }
    return metrics

class_metrics_phase1 = calculate_class_metrics(cm_phase1, class_names)

# Save class metrics to JSON
with open(f'{CONFIG["logs_dir"]}/resnet50_phase1_class_metrics.json', 'w') as f:
    json.dump(class_metrics_phase1, f, indent=4)

# Display top 5 best and worst classes by F1 score
f1_scores = {name: metrics['f1_score'] for name, metrics in class_metrics_phase1.items()}
sorted_f1 = sorted(f1_scores.items(), key=lambda x: x[1], reverse=True)

print("\nTop 5 Best Classes (by F1-Score):")
for class_name, f1 in sorted_f1[:5]:
    metrics = class_metrics_phase1[class_name]
    print(f"  {class_name:40s}: F1={f1:.4f}, Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}")

print("\nTop 5 Worst Classes (by F1-Score):")
for class_name, f1 in sorted_f1[-5:]:
    metrics = class_metrics_phase1[class_name]
    print(f"  {class_name:40s}: F1={f1:.4f}, Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}")


## 10. Plot Training History and Evaluation Metrics

In [ ]:
# Plot training history - Loss and Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(resnet50_history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(resnet50_history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('ResNet50 Phase 1: Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Accuracy
axes[1].plot(resnet50_history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(resnet50_history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('ResNet50 Phase 1: Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["plots_dir"]}/resnet50_phase1_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training history plot saved")

# Plot Confusion Matrix for Phase 1
plt.figure(figsize=(16, 14))
sns.heatmap(cm_phase1, annot=False, fmt='d', cmap='Blues', cbar=True,
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
plt.title('ResNet50 Phase 1: Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f'{CONFIG["confusion_matrix_dir"]}/resnet50_phase1_cm.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix plot saved")

# Plot class-wise F1 scores
f1_scores_sorted = dict(sorted(f1_scores.items(), key=lambda x: x[1]))
plt.figure(figsize=(12, 8))
colors = ['red' if f1 < 0.2 else 'orange' if f1 < 0.5 else 'green' for f1 in f1_scores_sorted.values()]
plt.barh(range(len(f1_scores_sorted)), list(f1_scores_sorted.values()), color=colors)
plt.yticks(range(len(f1_scores_sorted)), list(f1_scores_sorted.keys()), fontsize=8)
plt.xlabel('F1-Score', fontsize=12)
plt.title('ResNet50 Phase 1: Class-wise F1-Scores', fontsize=14, fontweight='bold')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CONFIG["plots_dir"]}/resnet50_phase1_f1_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print("Class-wise F1 scores plot saved")


## 11. Fine-Tuning Strategy: Unfreeze and Train with Lower Learning Rate

In [ ]:
# Fine-tuning strategy: Unfreeze last N layers of ResNet50 base model
print("\n" + "=" * 80)
print("PHASE 2: FINE-TUNING STRATEGY")
print("=" * 80)

# Count base model layers
base_model_layers = [layer for layer in resnet50_model.layers if 'resnet50' in layer.name]
print(f"\nBase model layer count: {len(base_model_layers)}")
print(f"Unfreezing last {CONFIG['num_unfreeze_layers']} layers for fine-tuning")
print(f"Phase 2 Learning Rate: {CONFIG['phase2_learning_rate']}")

# Unfreeze last N layers using the utility function
unfreeze_model_layers(
    resnet50_model,
    num_layers=CONFIG['num_unfreeze_layers'],
    learning_rate=CONFIG['phase2_learning_rate'],
    optimizer=CONFIG['optimizer']
)

# Verify which layers are trainable
trainable_layers = [layer.name for layer in resnet50_model.layers if layer.trainable]
print(f"\nTrainable layers: {len(trainable_layers)}")
print(f"Total parameters: {resnet50_model.count_params():,}")

# Check trainable parameter count
total_params = sum([tf.size(w).numpy() for w in resnet50_model.trainable_weights])
print(f"Trainable parameters: {total_params:,}")


## 12. Phase 2 Training (Fine-Tuning with Unfrozen Layers)

In [ ]:
# Train Phase 2 (fine-tuning with unfrozen layers)
print("\n" + "=" * 80)
print("PHASE 2 TRAINING: ResNet50 with Fine-Tuning")
print("=" * 80)
print(f"Epochs: {CONFIG['phase2_epochs']}")
print(f"Learning Rate: {CONFIG['phase2_learning_rate']}")
print(f"Batch Size: {CONFIG['batch_size']}")
print("=" * 80)

phase2_callbacks = create_callbacks('phase2')

resnet50_history_phase2 = resnet50_model.fit(
    X_train, y_train,
    epochs=CONFIG['phase2_epochs'],
    batch_size=CONFIG['batch_size'],
    validation_data=(X_val, y_val),
    class_weight=class_weight,
    callbacks=phase2_callbacks,
    verbose=1
)

# Save phase 2 training history
history_dict_phase2 = {
    'loss': [float(x) for x in resnet50_history_phase2.history['loss']],
    'accuracy': [float(x) for x in resnet50_history_phase2.history['accuracy']],
    'val_loss': [float(x) for x in resnet50_history_phase2.history['val_loss']],
    'val_accuracy': [float(x) for x in resnet50_history_phase2.history['val_accuracy']]
}

with open(f'{CONFIG["logs_dir"]}/resnet50_phase2_history.json', 'w') as f:
    json.dump(history_dict_phase2, f, indent=4)

print("\n" + "=" * 80)
print("PHASE 2 TRAINING COMPLETED")
print("=" * 80)
print(f"Final Train Loss: {resnet50_history_phase2.history['loss'][-1]:.4f}")
print(f"Final Train Accuracy: {resnet50_history_phase2.history['accuracy'][-1]:.4f}")
print(f"Final Val Loss: {resnet50_history_phase2.history['val_loss'][-1]:.4f}")
print(f"Final Val Accuracy: {resnet50_history_phase2.history['val_accuracy'][-1]:.4f}")


## 13. Evaluate Phase 2 Model and Compare with Phase 1

In [ ]:
# Evaluate Phase 2 model on test set
print("\n" + "=" * 80)
print("PHASE 2 EVALUATION ON TEST SET")
print("=" * 80)

metrics_phase2 = evaluate_model(
    resnet50_model,
    X_test,
    y_test,
    class_names=class_names
)

print("\nPhase 2 Test Set Metrics:")
print(f"  Accuracy: {metrics_phase2['accuracy']:.4f}")
print(f"  Macro Precision: {metrics_phase2['macro_precision']:.4f}")
print(f"  Macro Recall: {metrics_phase2['macro_recall']:.4f}")
print(f"  Macro F1-Score: {metrics_phase2['macro_f1']:.4f}")
print(f"  Weighted F1-Score: {metrics_phase2['weighted_f1']:.4f}")

cm_phase2 = metrics_phase2['confusion_matrix']

# Compare Phase 1 vs Phase 2
print("\n" + "=" * 80)
print("PHASE 1 vs PHASE 2 COMPARISON")
print("=" * 80)
print(f"{'Metric':<25} {'Phase 1':<15} {'Phase 2':<15} {'Improvement':<15}")
print("-" * 70)

metrics_to_compare = [
    ('Accuracy', 'accuracy'),
    ('Macro Precision', 'macro_precision'),
    ('Macro Recall', 'macro_recall'),
    ('Macro F1-Score', 'macro_f1'),
    ('Weighted F1-Score', 'weighted_f1')
]

for metric_name, metric_key in metrics_to_compare:
    phase1_val = metrics_phase1[metric_key]
    phase2_val = metrics_phase2[metric_key]
    improvement = phase2_val - phase1_val
    improvement_pct = (improvement / phase1_val * 100) if phase1_val > 0 else 0
    
    print(f"{metric_name:<25} {phase1_val:<15.4f} {phase2_val:<15.4f} {improvement:+.4f} ({improvement_pct:+.1f}%)")

# Plot comparison of training histories
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot Phase 1 Loss
axes[0, 0].plot(resnet50_history.history['loss'], label='Training', linewidth=2)
axes[0, 0].plot(resnet50_history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('Phase 1: Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot Phase 1 Accuracy
axes[0, 1].plot(resnet50_history.history['accuracy'], label='Training', linewidth=2)
axes[0, 1].plot(resnet50_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 1].set_title('Phase 1: Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot Phase 2 Loss
axes[1, 0].plot(resnet50_history_phase2.history['loss'], label='Training', linewidth=2)
axes[1, 0].plot(resnet50_history_phase2.history['val_loss'], label='Validation', linewidth=2)
axes[1, 0].set_title('Phase 2: Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot Phase 2 Accuracy
axes[1, 1].plot(resnet50_history_phase2.history['accuracy'], label='Training', linewidth=2)
axes[1, 1].plot(resnet50_history_phase2.history['val_accuracy'], label='Validation', linewidth=2)
axes[1, 1].set_title('Phase 2: Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["plots_dir"]}/resnet50_phase1_vs_phase2_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPhase 1 vs Phase 2 comparison plot saved")

# Save confusion matrix for Phase 2
np.save(f'{CONFIG["confusion_matrix_dir"]}/resnet50_phase2_cm.npy', cm_phase2)
print(f"Phase 2 confusion matrix saved")


## 14. Save and Load Models

In [ ]:
# Save final ResNet50 model
print("\n" + "=" * 80)
print("SAVING MODELS")
print("=" * 80)

# Save as .h5 format (includes architecture and weights)
final_model_path = f'{CONFIG["models_dir"]}/resnet50_final.h5'
resnet50_model.save(final_model_path)
print(f"✓ Final model saved: {final_model_path}")

# Save as SavedModel format (recommended for production)
savedmodel_path = f'{CONFIG["models_dir"]}/resnet50_final_savedmodel'
resnet50_model.save(savedmodel_path, save_format='tf')
print(f"✓ SavedModel saved: {savedmodel_path}")

# Save model configuration and metadata
metadata = {
    'architecture': 'ResNet50',
    'num_classes': CONFIG['num_classes'],
    'input_shape': CONFIG['input_shape'],
    'total_parameters': int(resnet50_model.count_params()),
    'trainable_parameters': int(sum([tf.size(w).numpy() for w in resnet50_model.trainable_weights])),
    'phase1_epochs': CONFIG['phase1_epochs'],
    'phase2_epochs': CONFIG['phase2_epochs'],
    'batch_size': CONFIG['batch_size'],
    'final_accuracy': float(metrics_phase2['accuracy']),
    'final_loss': float(resnet50_history_phase2.history['val_loss'][-1]),
    'class_names': class_names
}

with open(f'{CONFIG["models_dir"]}/resnet50_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"✓ Model metadata saved")

# Load and verify model
print("\n" + "=" * 80)
print("VERIFYING SAVED MODEL")
print("=" * 80)

loaded_model = keras.models.load_model(final_model_path)
print(f"✓ Model loaded successfully")
print(f"  Loaded model input shape: {loaded_model.input_shape}")
print(f"  Loaded model output shape: {loaded_model.output_shape}")

# Test prediction with loaded model
sample_predictions = loaded_model.predict(X_test[:5], verbose=0)
print(f"✓ Model predictions verified")
print(f"  Sample prediction shape: {sample_predictions.shape}")
print(f"  Sample prediction values (first image, top 5 classes):")
top_5_indices = np.argsort(sample_predictions[0])[::-1][:5]
for idx in top_5_indices:
    print(f"    {class_names[idx]}: {sample_predictions[0][idx]:.4f}")

print("\n" + "=" * 80)
print("RESNET50 TRANSFER LEARNING TRAINING COMPLETE")
print("=" * 80)
print(f"\nFinal Metrics:")
print(f"  Test Accuracy: {metrics_phase2['accuracy']:.4f}")
print(f"  Test F1-Score (Weighted): {metrics_phase2['weighted_f1']:.4f}")
print(f"  Total training time: ~{CONFIG['phase1_epochs'] + CONFIG['phase2_epochs']} epochs")
